<a href="https://colab.research.google.com/github/daspushpita/emotion-mechanisms-llm/blob/main/notebooks/colab_extract_activations_7b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Emotion Activation Extraction — Qwen2.5-7B-Instruct

In [1]:
!pip install -q transformers accelerate h5py huggingface_hub

In [2]:
import os, sys

REPO_DIR = '/content/emotion-mechanisms-llm'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/daspushpita/emotion-mechanisms-llm.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

for p in [f'{REPO_DIR}/src', f'{REPO_DIR}/scripts']:
    if p not in sys.path:
        sys.path.insert(0, p)

Already up to date.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Patch config BEFORE importing build_emotion_vectors so its module-level
# `from emotion_mechanisms.config import ...` picks up the Drive paths.
from pathlib import Path
import emotion_mechanisms.config as cfg

DRIVE_ROOT = Path('/content/drive/MyDrive/emotion-mechanisms-llm')  # edit if needed

cfg.EMOTIONAL_STORIES_DATASET = DRIVE_ROOT / 'datasets/processed/emotional_stories_qwen32B_v1.jsonl'
cfg.NEUTRAL_STORIES_DATASET   = DRIVE_ROOT / 'datasets/processed/neutral_stories_qwen32B_v1.jsonl'
cfg.ACTIVATIONS_PATH          = DRIVE_ROOT / 'results/activations/activations_7b.h5'
cfg.ANALYSIS_MODEL_7B         = 'Qwen/Qwen2.5-7B-Instruct'

assert cfg.EMOTIONAL_STORIES_DATASET.exists(), f'Missing: {cfg.EMOTIONAL_STORIES_DATASET}'
assert cfg.NEUTRAL_STORIES_DATASET.exists(),   f'Missing: {cfg.NEUTRAL_STORIES_DATASET}'
cfg.ACTIVATIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Datasets found. Output ->', cfg.ACTIVATIONS_PATH)

Datasets found. Output -> /content/drive/MyDrive/emotion-mechanisms-llm/results/activations/activations_7b.h5


In [5]:
from huggingface_hub import notebook_login
notebook_login()

In [6]:
# Import after patching config so the module picks up Drive paths
import build_emotion_vectors

# Set max_stories=3 to smoke-test the pipeline quickly.
# Remove the argument (or set None) for the full run.
build_emotion_vectors.main(max_stories=3)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [7]:
import h5py
with h5py.File(cfg.ACTIVATIONS_PATH, 'r') as f:
    print('Top-level keys:', list(f.keys()))
    print('Emotions stored:', list(f['emotional'].keys()))
    first_layer = list(f['emotional/happy'].keys())[0]
    print(f'emotional/happy/{first_layer} shape:', f[f'emotional/happy/{first_layer}'].shape)

Top-level keys: ['emotional', 'neutral']
Emotions stored: ['afraid', 'angry', 'calm', 'desperate', 'guilty', 'happy', 'inspired', 'loving', 'nervous', 'proud', 'sad', 'surprised']
emotional/happy/layer_0 shape: (3, 3584)
